Think of it is:

* State : The information that flows through the graph.
* Nodes : The workers that process the information.
* Edges : The roads that decide where the information goes next.

Without edges, all the nodes would exist, but they wouldn't know how to communicate with each other.

#### Edge
A Edge is a connection between two nodes.
it tells a langgraph ``After completing this node, which node should run next?``

Example: we have 3 nodes 
1. Read input
2. calculate result
3. show output

without edges they are just three separate functions.

Read Input

Calculate Result

Show Output

There is no relationship between the nodes as these are not connected , these will do individual tasks.

Now Connect with the edges : Now they become a workflow, the arrow represents the direction (path) towards the next action and conenction between the nodes so that they can share the updated state (information each other), these are connected to each other and there is relation exists between the nodes.

``` markdown
Read Input
      │
      ▼
Calculate Result
      │
      ▼
Show Output
```

#### Real world Analogy

suppose you will go to a hospital there you have to go with flow like first we should visit reception then doctor then pharmacy exit. how do you know where to go?

There are paths connecting each department.

``` markdown
    Reception
      │
      ▼
    Doctor
      │
      ▼ 
    Pharmacy
      │
      ▼
     Exit
```





#### Types of Edges
Langgraph mainly uses 3 edges
1. simple Edge
2. conditional Edge
3. Dynamic Edge



#### Simple Edge
A simple edge always goes to the same next node. it never asks questions and never checks conditons.
it simply says ```node A --> node B ```

suppose we have ```read question --> generate answer --> return answer.```

Graph

``` markdown
START
   │
   ▼
Read Question
   │
   ▼
Generate Answer
   │
   ▼
Return Answer
   │
   ▼
  END
```

* A simple edge is a fixed, unconditional connection from one node to the next. Every time node A finishes, execution always moves to node B — no conditions, no decisions.

![alt text](image.png)


In [ ]:
## Simple Node Example
# imagine we have 3 nodes
def read_question(state):
    return state

def generate_answer(state):
    return state

def return_answer(state):
    return state

# create a path and connect between the nodes : for this we need create a edges

graph.add_edge("read_question", "generate_answer")

graph.add_edge("generate_answer", "return_answer")

In [ ]:
### Simple Node 
from typing import TypedDict

# define the state scheam
class SimpleState(TypedDict):
    raw_text:str
    cleaned:str
    word_count:int

# 1st node : clean the text 
def clean_node(state:SimpleState)->dict:
    cleaned = state["raw_text"].strip().lower()
    print(f"[cleaned node] '{state['raw_text']}'->'{cleaned}'")
    return {"cleaned":cleaned}

# 2nd node count the word count
def count_node(state: SimpleState) -> dict:
    count = len(state["cleaned"].split())
    print(f"  [count_node]    word_count = {count}")
    return {"word_count": count}

# 3rd node  : validate the node
def validate_node(state: SimpleState) -> dict:
    valid = state["word_count"] >= 3
    print(f"  [validate_node] valid = {valid}  (need >= 3 words)")
    return {}

from langgraph.graph import StateGraph, START, END


builder = StateGraph(SimpleState)

builder.add_node("clean_node",    clean_node)
builder.add_node("count_node",    count_node)
builder.add_node("validate_node", validate_node)

builder.set_entry_point("clean_node")          # same as add_edge(START, "clean_node")

builder.add_edge("clean_node",    "count_node")
builder.add_edge("count_node",    "validate_node")
builder.add_edge("validate_node", END)

graph1 = builder.compile()


In [6]:
# test the node
test_inputs_1 = [
    "  Hello World from LangGraph  ",   # 4 words — valid
    "  Hi  ",                            # 1 word  — invalid
]

for text in test_inputs_1:
    print(f"\n  Input: '{text.strip()}'")
    result = graph1.invoke({"raw_text": text, "cleaned": "", "word_count": 0})
    print(f"  Final state: cleaned='{result['cleaned']}' | word_count={result['word_count']}")


  Input: 'Hello World from LangGraph'
[cleaned node] '  Hello World from LangGraph  '->'hello world from langgraph'
  [count_node]    word_count = 4
  [validate_node] valid = True  (need >= 3 words)
  Final state: cleaned='hello world from langgraph' | word_count=4

  Input: 'Hi'
[cleaned node] '  Hi  '->'hi'
  [count_node]    word_count = 1
  [validate_node] valid = False  (need >= 3 words)
  Final state: cleaned='hi' | word_count=1


#### Conditional Edge

Here in Conditional edges , sometimes, the next node depends on a condition. instead of ```always go here``` we say 
``` if x --> Go here else --> go there``` This is a conditional edge. 

Example:

Suppose a login system.
``` markdown
Enter Password
↓
Password Correct?
YES → Dashboard
NO → Try Again
```

Graph
``` markdown
              Enter Password
                     │
                     ▼
             Check Password
               /         \
              /           \
           Correct      Incorrect
             │              │
             ▼              ▼
       Dashboard        Try Again
```

The edge decides which path to follow. 

In [7]:
# python concept
# Routing function 
def route(state):
    if state["intent"] == "question":
        return "answer_node"
    return "exit_node"

# This function decides which edge to follow.

A conditional edge runs a router function after a node finishes. The router reads the state and returns a string that names the next node. LangGraph jumps to that node.

![alt text](image-1.png)

In [8]:
# =============================================================================
# 2. CONDITIONAL EDGES
#    add_conditional_edges(source, router_fn, mapping)
#    Router reads state, returns a key, LangGraph picks the next node.
#
#    Flow:  check_node → [route] → publish_node  (if valid)
#                               → reject_node   (if not valid)
# =============================================================================
# separator("2. CONDITIONAL EDGES")
 
class CondState(TypedDict):
    text:       str
    word_count: int
    is_valid:   bool
    outcome:    str
 
def check_node(state: CondState) -> dict:
    count = len(state["text"].split())
    valid = count >= 3
    print(f"  [check_node]   words={count} | is_valid={valid}")
    return {"word_count": count, "is_valid": valid}
 
def publish_node(state: CondState) -> dict:
    print(f"  [publish_node] ✓ Accepted and published")
    return {"outcome": "published"}
 
def reject_node(state: CondState) -> dict:
    print(f"  [reject_node]  ✗ Rejected — too short")
    return {"outcome": "rejected"}
 
# Router function — returns a string key, not a node name
def route_after_check(state: CondState) -> str:
    return "publish" if state["is_valid"] else "reject"
 
builder = StateGraph(CondState)
builder.add_node("check_node",   check_node)
builder.add_node("publish_node", publish_node)
builder.add_node("reject_node",  reject_node)
builder.set_entry_point("check_node")
builder.add_conditional_edges(
    "check_node",          # source node
    route_after_check,     # router function
    {                      # mapping: key → node name
        "publish": "publish_node",
        "reject":  "reject_node",
    }
)
builder.add_edge("publish_node", END)
builder.add_edge("reject_node",  END)
graph2 = builder.compile()
 
# --- Test inputs ---
test_inputs_2 = [
    "Hi",                              # 1 word  → rejected
    "LangGraph is really powerful",    # 4 words → published
]
for text in test_inputs_2:
    print(f"\n  Input: '{text}'")
    result = graph2.invoke({"text": text, "word_count": 0, "is_valid": False, "outcome": ""})
    print(f"  Final state: outcome='{result['outcome']}'")
 
 


  Input: 'Hi'
  [check_node]   words=1 | is_valid=False
  [reject_node]  ✗ Rejected — too short
  Final state: outcome='rejected'

  Input: 'LangGraph is really powerful'
  [check_node]   words=4 | is_valid=True
  [publish_node] ✓ Accepted and published
  Final state: outcome='published'


#### Dynamic Routing

Dynamic routing means the router function decides the path at runtime — based on live state values, not hard-coded logic. The set of possible destinations is declared upfront, but which one is chosen depends entirely on what is in the state when the node finishes.

![alt text](image-2.png)



In [9]:
#=====
# 3. DYNAMIC ROUTING
#    The router reads a value from state and returns it as the routing key.
#    The destination is not known until runtime — it depends on live state.
#
#    Flow:  classify_node → [route_by_type] → invoice_handler
#                                           → email_handler
#                                           → general_handler
# =============================================================================
#separator("3. DYNAMIC ROUTING")
 
class DocState(TypedDict):
    raw:      str
    doc_type: str
    result:   str
 
def classify_node(state: DocState) -> dict:
    raw = state["raw"].lower()
    if "invoice" in raw or "amount" in raw:
        doc_type = "invoice"
    elif "dear" in raw or "regards" in raw:
        doc_type = "email"
    else:
        doc_type = "general"
    print(f"  [classify_node]    doc_type='{doc_type}'")
    return {"doc_type": doc_type}
 
def invoice_handler(state: DocState) -> dict:
    print(f"  [invoice_handler]  Extracting line items and totals")
    return {"result": "Invoice parsed — extracted amounts and dates"}
 
def email_handler(state: DocState) -> dict:
    print(f"  [email_handler]    Parsing headers and body")
    return {"result": "Email parsed — extracted sender and subject"}
 
def general_handler(state: DocState) -> dict:
    print(f"  [general_handler]  Indexing as general document")
    return {"result": "General document indexed"}
 
# Dynamic: the router simply returns state["doc_type"] — no hardcoded logic
def route_by_type(state: DocState) -> str:
    return state["doc_type"]   # value written by classify_node drives the route
 
builder = StateGraph(DocState)
builder.add_node("classify_node",   classify_node)
builder.add_node("invoice_handler", invoice_handler)
builder.add_node("email_handler",   email_handler)
builder.add_node("general_handler", general_handler)
builder.set_entry_point("classify_node")
builder.add_conditional_edges(
    "classify_node",
    route_by_type,
    {"invoice": "invoice_handler", "email": "email_handler", "general": "general_handler"}
)
builder.add_edge("invoice_handler", END)
builder.add_edge("email_handler",   END)
builder.add_edge("general_handler", END)
graph3 = builder.compile()
 
# --- Test inputs ---
test_inputs_3 = [
    "Invoice #1042 — Total amount due: $4,500",
    "Dear team, please find the update. Regards, Subbu",
    "Quarterly strategy notes from the offsite",
]
for doc in test_inputs_3:
    label = doc[:50] + "..." if len(doc) > 50 else doc
    print(f"\n  Input: '{label}'")
    result = graph3.invoke({"raw": doc, "doc_type": "", "result": ""})
    print(f"  Final state: result='{result['result']}'")
 


  Input: 'Invoice #1042 — Total amount due: $4,500'
  [classify_node]    doc_type='invoice'
  [invoice_handler]  Extracting line items and totals
  Final state: result='Invoice parsed — extracted amounts and dates'

  Input: 'Dear team, please find the update. Regards, Subbu'
  [classify_node]    doc_type='email'
  [email_handler]    Parsing headers and body
  Final state: result='Email parsed — extracted sender and subject'

  Input: 'Quarterly strategy notes from the offsite'
  [classify_node]    doc_type='general'
  [general_handler]  Indexing as general document
  Final state: result='General document indexed'


#### Branching 

Branching sends execution down one path chosen from several options. Unlike fan-out (parallel), branching picks exactly one branch. The branches recombine at a shared downstream node.

![alt text](image-3.png)



In [10]:
# =============================================================================
# 4. BRANCHING
#    One conditional edge picks exactly ONE branch.
#    All branches converge at a shared downstream node (format_node).
#
#    Flow:  triage_node → [route_priority] → fast_path     ─┐
#                                          → standard_path ─┤→ format_node → END
#                                          → low_path      ─┘
# =============================================================================
# separator("4. BRANCHING (one path chosen, all converge)")
 
class TicketState(TypedDict):
    issue:    str
    priority: int
    response: str
 
def triage_node(state: TicketState) -> dict:
    print(f"  [triage_node]    issue='{state['issue']}' | priority=P{state['priority']}")
    return {}
 
def fast_path(state: TicketState) -> dict:
    print(f"  [fast_path]      P1 — paging on-call engineer NOW")
    return {"response": "P1 escalated to on-call engineer"}
 
def standard_path(state: TicketState) -> dict:
    print(f"  [standard_path]  P2 — added to sprint board")
    return {"response": "P2 ticket added to sprint"}
 
def low_path(state: TicketState) -> dict:
    print(f"  [low_path]       P3 — logged to backlog")
    return {"response": "P3 logged to backlog"}
 
def format_node(state: TicketState) -> dict:
    # All three branches converge here
    print(f"  [format_node]    → '{state['response']}'")
    return {}
 
def route_priority(state: TicketState) -> str:
    p = state["priority"]
    if p == 1: return "fast"
    if p == 2: return "standard"
    return "low"
 
builder = StateGraph(TicketState)
builder.add_node("triage_node",   triage_node)
builder.add_node("fast_path",     fast_path)
builder.add_node("standard_path", standard_path)
builder.add_node("low_path",      low_path)
builder.add_node("format_node",   format_node)
builder.set_entry_point("triage_node")
builder.add_conditional_edges(
    "triage_node", route_priority,
    {"fast": "fast_path", "standard": "standard_path", "low": "low_path"}
)
# All three branches converge at format_node
builder.add_edge("fast_path",     "format_node")
builder.add_edge("standard_path", "format_node")
builder.add_edge("low_path",      "format_node")
builder.add_edge("format_node",   END)
graph4 = builder.compile()
 
# --- Test inputs ---
test_inputs_4 = [
    ("Production DB down",  1),
    ("Login slow on mobile", 2),
    ("Update docs typo",     3),
]
for issue, priority in test_inputs_4:
    print(f"\n  Input: issue='{issue}' | priority=P{priority}")
    result = graph4.invoke({"issue": issue, "priority": priority, "response": ""})
    print(f"  Final state: response='{result['response']}'")


  Input: issue='Production DB down' | priority=P1
  [triage_node]    issue='Production DB down' | priority=P1
  [fast_path]      P1 — paging on-call engineer NOW
  [format_node]    → 'P1 escalated to on-call engineer'
  Final state: response='P1 escalated to on-call engineer'

  Input: issue='Login slow on mobile' | priority=P2
  [triage_node]    issue='Login slow on mobile' | priority=P2
  [standard_path]  P2 — added to sprint board
  [format_node]    → 'P2 ticket added to sprint'
  Final state: response='P2 ticket added to sprint'

  Input: issue='Update docs typo' | priority=P3
  [triage_node]    issue='Update docs typo' | priority=P3
  [low_path]       P3 — logged to backlog
  [format_node]    → 'P3 logged to backlog'
  Final state: response='P3 logged to backlog'


#### Multiple Paths
* multiple paths means , different workflows can eventually reach the same node.

* Fan-out means one node triggers multiple downstream nodes that run in parallel. LangGraph supports this by adding several edges from a single source. All parallel branches must finish before execution continues at the join node.

![alt text](image-4.png)

#### Real world analogy : Food delivery
``` markdown
Cash Payment
↓
Order Confirmed
```
or

``` markdown
online payment
↓
order confirmed
```
so here different methods of payment but the result will be one that is order confirmed.

Graph 

``` markdown
             Payment
          ┌─────┴─────┐
          ▼           ▼
      Cash        Online
          │           │
          └─────┬─────┘
                ▼
        Confirm Order
```
Here we can say like different paths but the same destination.


In [ ]:
from typing import Annotated
## python code 
# =============================================================================
# 5. MULTIPLE PATHS — Fan-out / Fan-in
#    Multiple add_edge() calls from one source = parallel execution.
#    All parallel branches must finish before the join node (embed_node) runs.
#    Uses Annotated reducers so concurrent writes to lists merge safely.
#
#    Flow:  ingest_node ──→ chunk_node    ─┐
#                      └──→ metadata_node ─┤→ embed_node → END
# =============================================================================
# separator("5. MULTIPLE PATHS (Fan-out / Fan-in)")
 
class PipelineState(TypedDict):
    raw_text: str
    chunks:   Annotated[list[str], add]   # reducer — parallel writes append
    metadata: Annotated[list[str], add]   # reducer — parallel writes append
    summary:  str
 
def ingest_node(state: PipelineState) -> dict:
    print(f"  [ingest_node]    Loaded: '{state['raw_text'][:45]}...'")
    return {}
 
def chunk_node(state: PipelineState) -> dict:
    """splits the documents into chunks """
    words  = state["raw_text"].split()
    chunks = [" ".join(words[i:i+4]) for i in range(0, len(words), 4)]
    print(f"  [chunk_node]     Created {len(chunks)} chunks  ← runs in parallel")
    return {"chunks": chunks}   # appended via 'add' reducer
 
def metadata_node(state: PipelineState) -> dict:
    """ It collects the metadata"""
    tags = ["word_count:" + str(len(state["raw_text"].split())), "lang:en"]
    print(f"  [metadata_node]  Tags: {tags}  ← runs in parallel")
    return {"metadata": tags}   # appended via 'add' reducer
 
def embed_node(state: PipelineState) -> dict:
    """Store in the vector database"""
    # Runs only after BOTH chunk_node AND metadata_node have finished
    print(f"  [embed_node]     Fan-in complete → chunks={len(state['chunks'])} | metadata={state['metadata']}")
    summary = f"Indexed {len(state['chunks'])} chunks with tags {state['metadata']}"
    return {"summary": summary}
 
builder = StateGraph(PipelineState)
builder.add_node("ingest_node",   ingest_node)
builder.add_node("chunk_node",    chunk_node)
builder.add_node("metadata_node", metadata_node)
builder.add_node("embed_node",    embed_node)
builder.set_entry_point("ingest_node")
# Fan-out: two edges from the same source # these nodes are run in parallel.
builder.add_edge("ingest_node",   "chunk_node")
builder.add_edge("ingest_node",   "metadata_node")
# Fan-in: both branches point to the same join node
builder.add_edge("chunk_node",    "embed_node")
builder.add_edge("metadata_node", "embed_node")
builder.add_edge("embed_node",    END) # all the state informtion is managed here 
graph5 = builder.compile() # when we run this app : the whole graph will excecute.
  
# --- Test input ---
test_input_5 = "LangGraph makes it easy to build stateful multi-step AI workflows"
print(f"\n  Input: '{test_input_5}'")
result = graph5.invoke({"raw_text": test_input_5, "chunks": [], "metadata": [], "summary": ""})
print(f"  Final state: summary='{result['summary']}'")

NameError: name 'add' is not defined

#### Edge Logic
Edge logic covers loops (routing back to an earlier node), cycle detection, and how to implement retry or self-correction patterns cleanly.
These kind of logics are implemented in a autonomus agents. where the agents are take decisions by themselves in that kind of applications this edge logic edges will be implemented.

![alt text](image-5.png)


Graph will look like 

``` markdown
                 START
                    │
                    ▼
             Detect Intent
      ┌────────┼─────────┬────────┐
      ▼        ▼         ▼        ▼
 Greeting   Question  Complaint  Goodbye
      │        │         │         │
      ▼        ▼         ▼         ▼
 Respond   Search KB  Create Ticket Exit
      │        │         │
      └────────┼─────────┘
               ▼
              END
```

#### How state, nodes and edges working together 

``` markdown
            STATE
              │
              ▼
        +-------------+
        |    Node     |
        | Processes   |
        +-------------+
              │
              ▼
      Updated State
              │
              ▼
            Edge
     Decides Next Node
              │
              ▼
        Next Node Runs
```

##### Summary Table

| Concept              | Meaning                                                            | Example                                                             |
| -------------------- | ------------------------------------------------------------------ | ------------------------------------------------------------------- |
| **Simple Edge**      | Always goes to the same next node                                  | Read Input → Process Input                                          |
| **Conditional Edge** | Chooses the next node based on a condition                         | Valid Login → Dashboard, Invalid Login → Retry                      |
| **Dynamic Routing**  | Determines the destination during execution from the current state | Route to Translation, Summarization, or Coding based on user intent |
| **Branching**        | One node splits into multiple possible paths                       | Complaint → Billing / Technical / Refund                            |
| **Multiple Paths**   | Different routes merge into a common node                          | Cash and Online payments both reach Order Confirmation              |
| **Edge Logic**       | Business rules that decide which edge to follow                    | Loan approval based on income and credit score                      |


In [12]:
## Edge logic code.

# =============================================================================
# 6. EDGE LOGIC — Loop with retry guard
#    A conditional edge can point back to a previous node, creating a loop.
#    ALWAYS guard loops with a max-retry counter in state.
#    Without it, a consistently failing node loops forever.
#
#    Flow:  llm_node → [check_quality] → END          (pass)
#                   ↑                 → llm_node       (retry — loops back)
#                   └─────────────────→ fallback_node  (max_retries exceeded)
# =============================================================================
# separator("6. EDGE LOGIC — Loop with retry guard")
 
class LoopState(TypedDict):
    prompt:        str
    response:      str
    quality_score: float
    retries:       int
    outcome:       str
 
def llm_node(state: LoopState) -> dict:
    attempt = state["retries"] + 1
    # Simulated: quality improves by 0.2 each attempt (0.7 → 0.9 on attempt 2)
    score    = round(min(0.5 + attempt * 0.2, 1.0), 2)
    response = f"Attempt #{attempt} answer to: '{state['prompt']}'"
    print(f"  [llm_node]       attempt={attempt} | quality_score={score}")
    return {"response": response, "quality_score": score, "retries": attempt}
 
def fallback_node(state: LoopState) -> dict:
    print(f"  [fallback_node]  Max retries hit — returning safe default")
    return {"response": "Sorry, could not generate a quality answer.", "outcome": "fallback"}
 
# Router — decides whether to loop, exit, or go to fallback
def check_quality(state: LoopState) -> str:
    score   = state["quality_score"]
    retries = state["retries"]
    if score >= 0.85:
        print(f"  [check_quality]  ✓ Score {score} ≥ 0.85 — PASS")
        return "pass"
    if retries >= 3:
        print(f"  [check_quality]  ✗ Max retries ({retries}) reached — FALLBACK")
        return "max_retries"
    print(f"  [check_quality]  ✗ Score {score} < 0.85 — RETRY")
    return "retry"
 
builder = StateGraph(LoopState)
builder.add_node("llm_node",      llm_node)
builder.add_node("fallback_node", fallback_node)
builder.set_entry_point("llm_node")
builder.add_conditional_edges(
    "llm_node",
    check_quality,
    {
        "pass":        END,              # quality met — done
        "retry":       "llm_node",      # loops back to same node
        "max_retries": "fallback_node", # safe exit
    }
)
builder.add_edge("fallback_node", END)
graph6 = builder.compile()
 
# --- Test input 1: passes on attempt 2 ---
print(f"\n  Input: prompt='Explain LangGraph edges'  (passes on attempt 2)")
result = graph6.invoke({
    "prompt": "Explain LangGraph edges",
    "response": "", "quality_score": 0.0, "retries": 0, "outcome": "success"
})
print(f"  Final state: retries={result['retries']} | response='{result['response']}'")
 
# --- Test input 2: force max-retries by starting at score 0.0 with max 3 ---
print(f"\n  Input: prompt='Intentionally bad prompt'  (hits fallback at retry 3)")
 
class LoopStateBad(TypedDict):
    prompt: str; response: str; quality_score: float; retries: int; outcome: str
 
def llm_node_bad(state):
    attempt = state["retries"] + 1
    score = 0.4  # never improves — forces fallback
    print(f"  [llm_node]       attempt={attempt} | quality_score={score} (always low)")
    return {"response": f"Bad attempt #{attempt}", "quality_score": score, "retries": attempt}
 
builder2 = StateGraph(LoopStateBad)
builder2.add_node("llm_node",      llm_node_bad)
builder2.add_node("fallback_node", fallback_node)
builder2.set_entry_point("llm_node")
builder2.add_conditional_edges(
    "llm_node", check_quality,
    {"pass": END, "retry": "llm_node", "max_retries": "fallback_node"}
)
builder2.add_edge("fallback_node", END)
graph6b = builder2.compile()
 
result = graph6b.invoke({
    "prompt": "Intentionally bad prompt",
    "response": "", "quality_score": 0.0, "retries": 0, "outcome": "success"
})
print(f"  Final state: retries={result['retries']} | outcome='{result['outcome']}'")
 
print("\n" + "="*60)
print("  All 6 edge examples completed successfully.")
print("="*60)
 


  Input: prompt='Explain LangGraph edges'  (passes on attempt 2)
  [llm_node]       attempt=1 | quality_score=0.7
  [check_quality]  ✗ Score 0.7 < 0.85 — RETRY
  [llm_node]       attempt=2 | quality_score=0.9
  [check_quality]  ✓ Score 0.9 ≥ 0.85 — PASS
  Final state: retries=2 | response='Attempt #2 answer to: 'Explain LangGraph edges''

  Input: prompt='Intentionally bad prompt'  (hits fallback at retry 3)
  [llm_node]       attempt=1 | quality_score=0.4 (always low)
  [check_quality]  ✗ Score 0.4 < 0.85 — RETRY
  [llm_node]       attempt=2 | quality_score=0.4 (always low)
  [check_quality]  ✗ Score 0.4 < 0.85 — RETRY
  [llm_node]       attempt=3 | quality_score=0.4 (always low)
  [check_quality]  ✗ Max retries (3) reached — FALLBACK
  [fallback_node]  Max retries hit — returning safe default
  Final state: retries=3 | outcome='fallback'

  All 6 edge examples completed successfully.
